# 預測分類模型之建置

In [1]:
from pathlib import Path
import pandas as pd
import joblib
import yaml
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..")
with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DATA_DIR = PROJECT_ROOT / config['data']['processed_dir']
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "default_flag"

def load_dataset(train_path, target_col=TARGET_COL, val_size=0.2, random_state=42):
    train_df = pd.read_csv(train_path)

    X = train_df.drop(columns=[target_col])
    y = train_df[target_col]
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=val_size,
        random_state=random_state,
        stratify=y
    )
    return X_train, y_train, X_val, y_val


## 模型一：隨機森林 (Random Forest)
隨機森林對離群值容忍度高，且不易過擬合，很適合拿來當基準模型 (Baseline)。

💡 小提示：因為違約樣本通常較少（類別不平衡），我們加上 class_weight='balanced' 來應對。

In [2]:
rf_train_path = DATA_DIR / "rf_train.csv"

if not rf_train_path.exists():
    available = [p.name for p in DATA_DIR.glob("*.csv")]
    raise FileNotFoundError(
        f"Missing RF training dataset. Expected '{rf_train_path.name}' in {DATA_DIR}.\n"
        f"Available CSV files: {available}"
    )

rf_X_train, rf_y_train, rf_X_val, rf_y_val = load_dataset(rf_train_path)


In [3]:
from sklearn.ensemble import RandomForestClassifier

# 1. 初始化模型
rf_model = RandomForestClassifier(
    n_estimators=100,       # 樹的棵數
    max_depth=10,           # 限制樹深防止過擬合
    class_weight='balanced',# 自動調整類別權重，照顧少數的違約樣本
    random_state=42
)

# 2. 訓練模型
rf_model.fit(rf_X_train, rf_y_train)

# 3. 預測（預測機率對你後續調整門檻、計算 ROC-AUC 非常重要）
rf_pred_proba = rf_model.predict_proba(rf_X_val)[:, 1] # 取得違約(1)的機率
rf_pred = rf_model.predict(rf_X_val)                   # 預測類別 (0 或 1)

# 4. 儲存模型
joblib.dump(rf_model, MODEL_DIR / "rf_model.joblib")

['..\\models\\rf_model.joblib']

## XGBoost 是目前處理表格資料（Tabular Data）的王者，通常能跑出極佳的效能。

小提示：針對偽陰性代價高的問題，我們可以使用 scale_pos_weight 來增加違約樣本的懲罰權重。

In [4]:
xgb_train_path = DATA_DIR / "xgb_train.csv"

if not xgb_train_path.exists():
    available = [p.name for p in DATA_DIR.glob("*.csv")]
    raise FileNotFoundError(
        f"Missing XGB training dataset. Expected '{xgb_train_path.name}' in {DATA_DIR}.\n"
        f"Available CSV files: {available}"
    )

xgb_X_train, xgb_y_train, xgb_X_val, xgb_y_val = load_dataset(xgb_train_path)


In [5]:
from xgboost import XGBClassifier

# 計算類別不平衡比例 (多數類別 / 少數類別)
ratio = (xgb_y_train == 0).sum() / (xgb_y_train == 1).sum()

# 1. 初始化模型
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=ratio, # 調整正負樣本權重，更重視違約(1)
    random_state=42,
    eval_metric='logloss'
)

# 2. 訓練模型（加入驗證集進行 Early Stopping，防止過擬合）
xgb_model.fit(
    xgb_X_train, xgb_y_train,
    eval_set=[(xgb_X_val, xgb_y_val)],
    verbose=False
)

# 3. 預測
xgb_pred_proba = xgb_model.predict_proba(xgb_X_val)[:, 1]
xgb_pred = xgb_model.predict(xgb_X_val)

# 4. 儲存模型
joblib.dump(xgb_model, MODEL_DIR / "xgb_model.joblib")

['..\\models\\xgb_model.joblib']

## 模型三：深度神經網路 (DNN / MLP)
利用 TensorFlow/Keras 建構標準的多層感知機（MLP）。深度學習對特徵縮放（如 StandardScaler）非常敏感，請確保特徵已轉換。

In [6]:
dnn_train_path = DATA_DIR / "dnn_train.csv"

if not dnn_train_path.exists():
    available = [p.name for p in DATA_DIR.glob("*.csv")]
    raise FileNotFoundError(
        f"Missing DNN training dataset. Expected '{dnn_train_path.name}' in {DATA_DIR}.\n"
        f"Available CSV files: {available}"
    )

dnn_X_train, dnn_y_train, dnn_X_val, dnn_y_val = load_dataset(dnn_train_path)


In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. 搭建網路架構
dnn_model = Sequential([
    # 輸入層 + 第一隱藏層（假設 X_train 有 N 個特徵）
    Dense(64, activation='relu', input_shape=(dnn_X_train.shape[1],)),
    Dropout(0.3),  # 隨機失活防止過擬合
    
    # 第二隱藏層
    Dense(32, activation='relu'),
    Dropout(0.3),
    
    # 輸出層（二分類任務，使用 Sigmoid 激活函數輸出 0~1 的機率）
    Dense(1, activation='sigmoid')
])

# 2. 編譯模型（定義損失函數與優化器）
dnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')] # 關注 AUC 指標
)

# 設定早停機制，若驗證集損失連續 5 個 epoch 沒改善就停止
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 3. 訓練模型
dnn_model.fit(
    dnn_X_train, dnn_y_train,
    validation_data=(dnn_X_val, dnn_y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# 4. 預測
dnn_pred_proba = dnn_model.predict(dnn_X_val).flatten()
# 預設以 0.5 為門檻轉換成 0 或 1（後續進入步驟六可再調整）
dnn_pred = (dnn_pred_proba > 0.5).astype(int)

# 5. 儲存模型
dnn_model.save(MODEL_DIR / "dnn_model.keras")

c:\Users\fromn\miniconda3\envs\dataMining\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.6786 - loss: 0.6140 - val_auc: 0.7393 - val_loss: 0.5685
Epoch 2/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7197 - loss: 0.5847 - val_auc: 0.7441 - val_loss: 0.5584
Epoch 3/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7296 - loss: 0.5740 - val_auc: 0.7465 - val_loss: 0.5544
Epoch 4/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7369 - loss: 0.5679 - val_auc: 0.7488 - val_loss: 0.5518
Epoch 5/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7434 - loss: 0.5630 - val_auc: 0.7491 - val_loss: 0.5499
Epoch 6/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7458 - loss: 0.5617 - val_auc: 0.7490 - val_loss: 0.5483
Epoch 7/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7469 - loss: 0.5578 - val_auc: 0.7500 - val_loss: 0.5492
Epoch 8/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7467 - loss: 0.5600 - val_auc: 0.7496 - val_loss: 0.5487
Epoch 9/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - au